In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
df = pd.read_csv(r"C:\Users\LENOVO\OneDrive\Desktop\Veri-Bilimi-Final-Projesi\Credit-risk.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\LENOVO\\OneDrive\\Desktop\\Veri-Bilimi-Final-Projesi\\Credit-risk.csv'

## Data Cleaning

### Fill missing employment length with 0 (assuming unemployed)

In [ ]:
df["person_emp_length"] =df["person_emp_length"].fillna(0)
df

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df[df["loan_int_rate"].isna()]

In [ ]:
df["loan_int_rate"].median()

### Fill missing interest rates with median to avoid skew from outliers
### +0.001 added to distinguish imputed values from natural median values

In [ ]:
df["loan_int_rate"] = df["loan_int_rate"].fillna(df["loan_int_rate"].median() + 0.001)
df[df["loan_int_rate"] == 10.919]

### Identifying outliers

In [ ]:
df = df[df["person_age"] < 100]
df

In [ ]:
df[df["person_emp_length"] > 60]
df = df[df["person_emp_length"] < 60]
df

In [ ]:
df["loan_to_income_ratio"] = df["loan_amnt"] / df["person_income"]
df

# Summary Statistics (EDA)

In [ ]:
print(f"Total Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print("\nBasic Statistics for Key Variables:")
print(df[['person_age', 'person_income', 'loan_amnt', 'loan_int_rate']].describe())

## Adding new columns
### Risk Level Column

In [ ]:
def risk_calc(row):
    # If borrower already defaulted, automatically High risk
    if row["loan_status"] == 1: return "High"
    # Loan-to-income ratio thresholds based on standard lending guidelines
    if row["loan_to_income_ratio"] > 0.5: return "High"
    if 0.3 <= row["loan_to_income_ratio"] <= 0.5: return "Medium"
    if row["loan_to_income_ratio"] < 0.3: return "Low"

In [ ]:
df["risk"] = df.apply(risk_calc, axis = 1)

In [ ]:
df

### Job Stability column

In [ ]:
df["emp_stability_ratio"] = df["person_emp_length"] / (df["person_age"] - 18)

### Loan Burden Index

In [ ]:
df["loan_burden_index"] = df['loan_to_income_ratio'] * df['loan_int_rate']

## Adding a risk-adjusted income column

### Reduce income by 30% for borrowers with prior defaults to reflect their higher effective risk

In [ ]:
df["risk_adjusted_income"] = df.apply(lambda x: x["person_income"] * 0.7 if x["cb_person_default_on_file"] == "Y" else x["person_income"], axis=1)

In [ ]:
df

### Categorizing Customers


In [ ]:
def categorize_customers(row):
     # Young borrowers (<30) with above-median income and short employment
    if row["person_age"] < 30 and row["person_income"] > np.median(df["person_income"]) and row["person_emp_length"] <= 10.0: return "Young & High Potential"
    # Borrowers with high debt burden or prior default history
    if row["loan_to_income_ratio"] > 0.5 or row["cb_person_default_on_file"] == "Y": return "High Risk"
    # Experienced borrowers with modest income and manageable debt
    if row["person_income"] < 100000 and row["cb_person_cred_hist_length"] >= 10 and row["loan_amnt"] < 100000: return "Financial Giant"
    else: return "Standard"

In [ ]:
df["customer_category"] = df.apply(categorize_customers, axis=1)

### Credit Maturity Index

In [ ]:
df['credit_maturity_index'] = df["cb_person_cred_hist_length"] / df["person_age"]

### Heuristic Scoring (Red-Flag)

In [ ]:
df["red-flag"] = ((df["cb_person_default_on_file"] == "Y") & (df["loan_amnt"] > 1000 )).astype(int)
df

# 1. CORRELATION MATRIX AND HEATMAP

In [ ]:
# Select only numerical columns (exclude strings and objects)
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

# Create heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix (Pearson)', fontsize=14, fontweight='bold')
plt.xlabel('Variables')
plt.ylabel('Variables')
plt.tight_layout()
plt.show()

# Identify high correlations
print("\nHigh Correlations (|r| > 0.5):")
high_corr = correlation_matrix.unstack()
high_corr = high_corr[high_corr > 0.5]
high_corr = high_corr[high_corr < 1.0]  # Exclude self-correlation (1.0)
print(high_corr)

# 2. RISK LEVEL DISTRIBUTION

In [ ]:
risk_counts = df['risk'].value_counts()
print(risk_counts)
print(f"\nRisk Distribution (%):")
print(df['risk'].value_counts(normalize=True) * 100)

# Create bar plot
plt.figure(figsize=(8, 6))
colors = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}
sns.countplot(data=df, x='risk', palette=colors, order=['Low', 'Medium', 'High'])
plt.title('Customer Risk Level Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Risk Level')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

# 3.AGE-INCOME Relationship

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['person_age'], df['person_income'],
                     alpha=0.5, c=df['loan_status'], cmap='RdYlGn_r', s=50)
plt.xlabel('Age (years)')
plt.ylabel('Income ($)')
plt.title('Age vs Income (Color: Loan Status)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Loan Status (0=Good, 1=Default)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
age_income_corr = df['person_age'].corr(df['person_income'])
print(f"Age-Income Correlation: {age_income_corr:.3f}")

# 4. LOAN INTEREST RATE DISTRIBUTION

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['loan_int_rate'], bins=40, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(df['loan_int_rate'].mean(), color='red', linestyle='--',
            linewidth=2, label=f"Mean: {df['loan_int_rate'].mean():.2f}%")
plt.axvline(df['loan_int_rate'].median(), color='green', linestyle='--',
            linewidth=2, label=f"Median: {df['loan_int_rate'].median():.2f}%")
plt.xlabel('Interest Rate (%)')
plt.ylabel('Frequency')
plt.title('Loan Interest Rate Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Interest Rate Statistics:")
print(f"  Min: {df['loan_int_rate'].min():.2f}%")
print(f"  Max: {df['loan_int_rate'].max():.2f}%")
print(f"  Mean: {df['loan_int_rate'].mean():.2f}%")
print(f"  Std Dev: {df['loan_int_rate'].std():.2f}%")

# 5. LOAN TO INCOME RATIO DISTRIBUTION

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='loan_to_income_ratio', kde=True, bins=40, color='lightcoral')
plt.xlabel('Loan to Income Ratio')
plt.ylabel('Frequency')
plt.title('Loan to Income Ratio Distribution (with KDE)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Relationship with risk level
print("\nLoan to Income Ratio by Risk Level:")
print(df.groupby('risk')['loan_to_income_ratio'].describe())

# ===== RESEARCH QUESTIONS & ANALYSIS =====

## RQ1: Credit History Length vs Default Rate

In [ ]:
print("""
RQ1: "How does credit history length affect loan default probability?"
    Hypothesis: Customers with longer credit history have lower default rates

RQ2: "Which loan purpose has the highest default risk, and how does it relate to interest rates?"
    Hypothesis: PERSONAL loans have higher default rates & higher interest rates

RQ3: "Which of the three engineered financial features best predicts
      loan default: employment stability, loan-to-income ratio,
      or loan burden index?"

Hypothesis: Employment stability is a strong predictor of default,
            possibly better than loan-to-income alone
""")

print("=" * 70)
print("ANSWERING RESEARCH QUESTION 1")
print("=" * 70)

print("\nDefault Rate by Credit History Length Groups:")
df['cred_hist_group'] = pd.cut(df['cb_person_cred_hist_length'],
                                bins=[0, 5, 10, 15, 20, 30],
                                labels=['0-5 years', '5-10 years', '10-15 years',
                                       '15-20 years', '20+ years'])

rq1_analysis = df.groupby('cred_hist_group')['loan_status'].agg(['count', 'sum', 'mean'])
rq1_analysis.columns = ['Total Customers', 'Defaults', 'Default Rate']
rq1_analysis['Default %'] = rq1_analysis['Default Rate'] * 100
print(rq1_analysis)

# Visualization for RQ1
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='cred_hist_group', y='loan_status', estimator=lambda x: (x.sum()/len(x))*100,
            palette='viridis')
plt.title('Default Rate by Credit History Length (RQ1)', fontsize=14, fontweight='bold')
plt.xlabel('Credit History Length')
plt.ylabel('Default Rate (%)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nRQ1 Finding (CORRECTED):")
print("→ Default rate DECREASES from 0-5 years to 15-20 years (22.5% → 20.8%)")
print("→ BUT default rate SPIKES UP at 20+ years (25.9%)")
print("→ Hypothesis PARTIALLY REJECTED: The relationship is NON-LINEAR!")
print("→ Longer credit history is protective... UNTIL 20+ years credit history")
print("→ This suggests other factors affect older borrowers differently")

## RQ2: Loan Purpose vs Default Rate and Interest Rate

In [ ]:
print("\n" + "=" * 70)
print("ANSWERING RESEARCH QUESTION 2")
print("=" * 70)

print("\nDefault Rate & Average Interest Rate by Loan Purpose:")
rq2_analysis = df.groupby('loan_intent').agg({
    'loan_status': ['count', 'sum', 'mean'],
    'loan_int_rate': 'mean'
}).round(4)

rq2_analysis.columns = ['Total Customers', 'Defaults', 'Default Rate', 'Avg Interest Rate']
rq2_analysis['Default %'] = rq2_analysis['Default Rate'] * 100
rq2_analysis = rq2_analysis.sort_values('Default %', ascending=False)
print(rq2_analysis)

# Visualization 1: Default Rate by Loan Purpose
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Default Rate
sns.barplot(data=df, x='loan_intent', y='loan_status',
            estimator=lambda x: (x.sum()/len(x))*100, ax=axes[0], palette='coolwarm')
axes[0].set_title('Default Rate by Loan Purpose (RQ2)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Loan Purpose')
axes[0].set_ylabel('Default Rate (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Average Interest Rate by Purpose
purpose_int_rate = df.groupby('loan_intent')['loan_int_rate'].mean().sort_values(ascending=False)
axes[1].barh(purpose_int_rate.index, purpose_int_rate.values, color='skyblue')
axes[1].set_title('Average Interest Rate by Loan Purpose', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Interest Rate (%)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## RQ3: Comparing Engineered Features as Default Predictors
### Which feature best predicts loan default: Employment Stability, Loan-to-Income Ratio, or Loan Burden Index?

In [ ]:
print("\n" + "=" * 70)
print("RQ2 FINDING & INTERPRETATION")
print("=" * 70)

print("""
FINDING #1: Highest Default Rate by Loan Purpose
──────────────────────────────────────────────────
1st: DEBTCONSOLIDATION → 28.59% default rate (HIGHEST!)
2nd: MEDICAL           → 26.70% default rate
3rd: HOMEIMPROVEMENT   → 26.10% default rate
4th: PERSONAL          → 19.88% default rate
5th: EDUCATION         → 17.22% default rate
6th: VENTURE           → 14.82% default rate (LOWEST!)

KEY INSIGHT: Our hypothesis was WRONG!
──────────────────────────────────────
Hypothesis: "PERSONAL loans have highest default rates"
Reality: DEBTCONSOLIDATION loans have highest default rates (28.59%)
         PERSONAL loans are actually MIDDLE-tier (19.88%)

FINDING #2: Interest Rate vs Default Rate Relationship
──────────────────────────────────────────────────────
Interest Rate Range: 10.95% - 11.18% (only 0.23% spread!)

Loan Purpose         Default Rate    Interest Rate    Correlation?
─────────────────────────────────────────────────────────────────
DEBTCONSOLIDATION    28.59% ↑        10.98% ↓         WEAK!
MEDICAL              26.70% ↑        11.05% ↑         Slight match
HOMEIMPROVEMENT      26.10% ↑        11.18% ↑↑        Good match
PERSONAL             19.88%          10.99%           No match
EDUCATION            17.22% ↓        10.95% ↓         Slight match
VENTURE              14.82% ↓        10.95% ↓         Good match

⚠️ CRITICAL FINDING: BANKS ARE NOT PRICING RISK CORRECTLY!
──────────────────────────────────────────────────────────
✗ DEBTCONSOLIDATION: HIGHEST default (28.59%) but LOWEST interest rate (10.98%)
✓ VENTURE: LOWEST default (14.82%) and LOWEST interest rate (10.95%)

This suggests:
1. The bank is underpricing DEBTCONSOLIDATION loans (too risky, too cheap!)
2. The bank is correctly pricing VENTURE loans (low risk, low rate)
3. Interest rates only vary by 0.23% despite 13.77% difference in default rates
   → This is a PRICING INEFFICIENCY!

WHY IS DEBTCONSOLIDATION SO RISKY?
──────────────────────────────────
People taking debt consolidation loans are already struggling:
- They have multiple debts they can't manage separately
- They're already in financial stress
- Higher likelihood of defaulting on consolidated loan
- Behavioral indicator of financial weakness

WHY IS VENTURE SO SAFE?
──────────────────────
People taking venture/business loans are typically:
- Entrepreneurs with business plans
- More financial literacy
- Better financial discipline
- Lower default risk (14.82% is lowest!)
""")

# Statistical Comparison
print("\nDEFAULT RATE SPREAD:")
min_default = rq2_analysis['Default %'].min()
max_default = rq2_analysis['Default %'].max()
spread = max_default - min_default
print(f"  Min (VENTURE):            {min_default:.2f}%")
print(f"  Max (DEBTCONSOLIDATION):  {max_default:.2f}%")
print(f"  Spread:                   {spread:.2f} percentage points!")

print("\nINTEREST RATE SPREAD:")
interest_by_purpose = df.groupby('loan_intent')['loan_int_rate'].mean()
print(f"  Min:  {interest_by_purpose.min():.2f}%")
print(f"  Max:  {interest_by_purpose.max():.2f}%")
print(f"  Spread: {interest_by_purpose.max() - interest_by_purpose.min():.2f}%")

print(f"\n⚠️ DEFAULT RATE VARIES BY {spread:.2f}% but INTEREST RATE ONLY VARIES BY {interest_by_purpose.max() - interest_by_purpose.min():.2f}%")
print("   This is a HUGE DISCREPANCY! The bank should charge more for risky loans.\n")

print("=" * 70)
print("RQ2 CONCLUSION:")
print("=" * 70)
print("""
✗ Hypothesis REJECTED: PERSONAL loans are NOT highest risk
✓ Finding: DEBTCONSOLIDATION loans have 2x higher default rate than VENTURE
✗ Banks NOT pricing risk correctly: High-risk loans have lower interest rates
✓ Implication: Better risk assessment could improve profitability

RECOMMENDATION FOR THE BANK:
→ Increase interest rates on DEBTCONSOLIDATION loans (currently 10.98%)
→ Reduce interest rates on VENTURE loans (currently 10.95%, already safe)
→ Use loan purpose as a KEY feature in predictive model
""")

In [ ]:
print("\n" + "=" * 70)
print("ANSWERING RESEARCH QUESTION 3")
print("=" * 70)

print("\nCorrelation with Default (loan_status):")
correlation_with_default = pd.DataFrame({
    'Variable': ['loan_to_income_ratio', 'emp_stability_ratio', 'loan_burden_index'],
    'Correlation with Default': [
        df['loan_to_income_ratio'].corr(df['loan_status']),
        df['emp_stability_ratio'].corr(df['loan_status']),
        df['loan_burden_index'].corr(df['loan_status'])
    ]
})
correlation_with_default = correlation_with_default.sort_values('Correlation with Default', ascending=False)
print(correlation_with_default)

# Visualization: Comparison of Predictors
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Employment Stability vs Default
axes[0].scatter(df['emp_stability_ratio'], df['loan_status'], alpha=0.3, s=20, color='steelblue')
axes[0].set_title('Employment Stability vs Default', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Employment Stability Ratio (emp_length / age)')
axes[0].set_ylabel('Loan Status (0=Good, 1=Default)')
axes[0].grid(True, alpha=0.3)

# Plot 2: Loan to Income vs Default
axes[1].scatter(df['loan_to_income_ratio'], df['loan_status'], alpha=0.3, s=20, color='orange')
axes[1].set_title('Loan to Income Ratio vs Default', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Loan to Income Ratio')
axes[1].set_ylabel('Loan Status (0=Good, 1=Default)')
axes[1].grid(True, alpha=0.3)

# Plot 3: Loan Burden Index vs Default
axes[2].scatter(df['loan_burden_index'], df['loan_status'], alpha=0.3, s=20, color='red')
axes[2].set_title('Loan Burden Index vs Default', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Loan Burden Index (ratio × interest rate)')
axes[2].set_ylabel('Loan Status (0=Good, 1=Default)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('RQ3: Comparing 3 Engineered Features as Default Predictors',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistical comparison
print("\nDefault Rate by Employment Stability Groups:")
df['emp_stability_group'] = pd.qcut(df['emp_stability_ratio'],
                                     q=4,
                                     labels=['Very Low', 'Low', 'Medium', 'High'],
                                     duplicates='drop')
rq3_analysis = df.groupby('emp_stability_group')['loan_status'].agg(['count', 'sum', 'mean'])
rq3_analysis.columns = ['Total', 'Defaults', 'Default Rate']
rq3_analysis['Default %'] = rq3_analysis['Default Rate'] * 100
print(rq3_analysis)

In [ ]:
print("\n" + "=" * 70)
print("RQ3 FINDING & INTERPRETATION")
print("=" * 70)

print("""
FINDING #1: Correlation Strength Comparison
─────────────────────────────────────────────
1st: loan_burden_index        → 0.4536 (STRONGEST!)
2nd: loan_to_income_ratio     → 0.3858
3rd: emp_stability_ratio      → -0.0696 (WEAKEST! Almost NO correlation!)

RANKING: loan_burden_index > loan_to_income_ratio >> emp_stability_ratio

KEY INSIGHT: Hypothesis REJECTED!
──────────────────────────────────
Hypothesis: "Employment stability is a better predictor than loan-to-income ratio"
Reality: Employment stability (r = -0.07) is TERRIBLE at predicting default!
         It's almost useless as a predictor (correlation near 0)

FINDING #2: Employment Stability vs Default Rate Pattern
─────────────────────────────────────────────────────────
Stability Level    Default Rate    Trend
─────────────────────────────────────────
Very Low          26.68% ↑↑↑       (HIGHEST - unstable people default more)
Low               22.11% ↑
Medium            19.70% ↓
High              18.42% ↓↓↓       (LOWEST - stable people default less)

Pattern: CLEAR INVERSE relationship! Higher stability = Lower defaults
         This DOES make logical sense!

⚠️ BUT... Why is correlation so weak (r = -0.07)?
──────────────────────────────────────────────
Answer: The correlation coefficient measures LINEAR relationship
        The employment stability effect is REAL (26.68% → 18.42%)
        But the relationship is WEAK because:

        1. Many other factors matter more (loan burden, income, credit history)
        2. Employment years alone don't capture financial stability
        3. A person with 10 years at one job may be less stable than
           someone with 5 years AND high income

FINDING #3: loan_burden_index is the BEST Predictor!
──────────────────────────────────────────────────────
Why? Because it COMBINES TWO FACTORS:

    loan_burden_index = (loan_to_income_ratio) × (loan_int_rate)

This captures:
    ✓ Loan amount relative to income (affordability)
    ✓ Interest rate (lender's risk assessment)
    ✓ Combined financial pressure

Example:
    Person A: ratio=0.3, rate=10% → burden=3.0   (comfortable)
    Person B: ratio=0.3, rate=15% → burden=4.5   (more stress)
    Person C: ratio=0.5, rate=10% → burden=5.0   (high stress)

The burden index captures the MULTIPLICATIVE effect of both factors!

COMPARISON: What Each Variable Tells Us
─────────────────────────────────────────
Variable                What It Measures              Correlation   Usefulness
────────────────────────────────────────────────────────────────────────────────
loan_burden_index      Total financial pressure      0.4536         ★★★★★ (BEST!)
                       (loan size × interest rate)

loan_to_income_ratio   Affordability relative        0.3858         ★★★★☆ (Good)
                       to income

emp_stability_ratio    Job tenure relative to age    -0.0696        ★☆☆☆☆ (Poor)
                       (weak predictor)
""")

# Statistical Analysis
print("\nDEFAULT RATE IMPACT BY EMPLOYMENT STABILITY:")
stability_stats = df.groupby('emp_stability_group')['loan_status'].agg(['count', 'sum', 'mean'])
stability_stats['Default %'] = stability_stats['mean'] * 100
default_spread_stability = stability_stats['Default %'].max() - stability_stats['Default %'].min()
print(f"  Range: {stability_stats['Default %'].min():.2f}% (High) to {stability_stats['Default %'].max():.2f}% (Very Low)")
print(f"  Spread: {default_spread_stability:.2f} percentage points")

print("\nDEFAULT RATE IMPACT BY LOAN-TO-INCOME RATIO:")
# Compare with loan to income (using the risk groups from earlier)
lti_stats = df.groupby('risk')['loan_status'].agg(['count', 'sum', 'mean'])
lti_stats['Default %'] = lti_stats['mean'] * 100
default_spread_lti = lti_stats['Default %'].max() - lti_stats['Default %'].min()
print(f"  Range: {lti_stats['Default %'].min():.2f}% to {lti_stats['Default %'].max():.2f}%")
print(f"  Spread: {default_spread_lti:.2f} percentage points")

print(f"\n✓ Loan-to-income ratio creates {default_spread_lti:.2f}% difference in default rates")
print(f"✓ Employment stability creates {default_spread_stability:.2f}% difference in default rates")
print(f"   (Both have practical impact, but loan burden matters MORE for the bank)")

print("\n" + "=" * 70)
print("RQ3 CONCLUSION:")
print("=" * 70)
print("""
✗ Hypothesis REJECTED: Employment stability is NOT a good predictor
✓ Finding: loan_burden_index is STRONGEST predictor (r=0.4536)
✓ Finding: loan_to_income_ratio is better than employment stability
✗ Finding: emp_stability_ratio is almost useless for prediction (r=-0.07)

KEY TAKEAWAY FOR MODELING:
→ Use loan_burden_index as PRIMARY feature (best correlation)
→ Use loan_to_income_ratio as SECONDARY feature (good correlation)
→ DO NOT rely on employment stability alone (too weak)
→ Consider COMBINING features for better predictions

BUSINESS INSIGHT:
The bank should focus on loan burden (size + interest rate), not just
employment tenure. A stable job with a massive loan is riskier than
an unstable job with a small loan.

RECOMMENDATION:
When building the classification model, prioritize:
  1. loan_burden_index (0.45 correlation)
  2. loan_to_income_ratio (0.39 correlation)
  3. Other factors (credit history, loan purpose, etc.)
  4. Employment stability (weak predictor, use with caution)
""")

# ===== CLASSIFICATION MODEL =====

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)


In [ ]:
print("=" * 70)
print("STEP 4: LOGISTIC REGRESSION MODEL")
print("=" * 70)

# ── FEATURE SELECTION ──
print("\nStep 4.1: Selecting Features...")

# Encode categorical variable (loan_intent) into numbers
df_model = pd.get_dummies(df, columns=['loan_intent'], drop_first=True)

# Select features based on RQ findings
features = [
    'loan_burden_index',          # RQ3: strongest predictor
    'loan_to_income_ratio',       # RQ3: second strongest
    'loan_int_rate',              # interest rate
    'cb_person_cred_hist_length', # RQ1: credit history
    'person_age',                 # RQ1: age factor
    'emp_stability_ratio',        # RQ3: weak but included
    'loan_intent_EDUCATION',      # RQ2: loan purpose (encoded)
    'loan_intent_HOMEIMPROVEMENT',
    'loan_intent_MEDICAL',
    'loan_intent_PERSONAL',
    'loan_intent_VENTURE'
]

X = df_model[features]
y = df_model['loan_status']

print(f"Features selected: {len(features)}")
print(f"Total samples: {len(X)}")
print(f"Default rate in dataset: {y.mean()*100:.2f}%")

# ── TRAIN/TEST SPLIT ──
print("\nStep 4.2: Splitting Data (80% train, 20% test)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples:  {len(X_train)}")
print(f"Testing samples:   {len(X_test)}")

# ── SCALING ──
print("\nStep 4.3: Scaling Features (StandardScaler)...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features scaled successfully!")

# ── TRAIN MODEL ──
print("\nStep 4.4: Training Logistic Regression Model...")
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)
print("Model trained successfully!")

# ── PREDICTIONS ──
print("\nStep 4.5: Making Predictions...")
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

## ── MODEL EVALUATION ──

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\nAccuracy:  {accuracy*100:.2f}%")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=['Good Loan (0)', 'Default (1)']))

## Confusion matrix plot

In [ ]:
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Good', 'Predicted Default'],
            yticklabels=['Actual Good', 'Actual Default'])
plt.title('Confusion Matrix - Logistic Regression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ROC curve plot

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Feature importance plot

In [ ]:
importance_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("Feature Importance (Model Coefficients):")
print(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Coefficient'], color='steelblue')
plt.xlabel('Coefficient Value')
plt.title('Feature Importance - Logistic Regression Coefficients',
          fontsize=14, fontweight='bold')
plt.axvline(x=0, color='red', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
print("=" * 70)
print("MODEL INTERPRETATION & LIMITATIONS")
print("=" * 70)
print("""
RESULTS SUMMARY:
────────────────
✅ Accuracy: 83.61% — Model correctly classifies 83% of loans
✅ ROC-AUC: 0.8323 — Strong ability to distinguish good vs default loans

KEY FINDING — CLASS IMBALANCE PROBLEM:
────────────────────────────────────────
⚠️ Default Recall: 41% — Model MISSES 59% of actual defaults!
⚠️ 834 out of 1421 defaults were NOT caught
⚠️ This is critical: in banking, missing defaults is very costly!

WHY IS DEFAULT RECALL LOW?
───────────────────────────
Dataset is IMBALANCED:
  Good Loans: 78.18% of data (majority class)
  Defaults:   21.82% of data (minority class)
The model is biased toward predicting "Good Loan"
because that's the most common outcome!

FEATURE IMPORTANCE INSIGHTS:
──────────────────────────────
✅ loan_to_income_ratio (coef=+2.05): STRONGEST positive predictor
   → Higher ratio = much more likely to default
✅ loan_int_rate (coef=+1.43): Second strongest positive predictor
   → Higher interest rate = higher default probability
⚠️ loan_burden_index (coef=-1.20): Negative due to MULTICOLLINEARITY
   → It overlaps with loan_to_income_ratio and loan_int_rate
   → Model compensates with negative coefficient to avoid double counting
✅ loan_intent_VENTURE (coef=-0.41): Confirms RQ2 finding
   → VENTURE loans REDUCE default risk (consistent with 14.82% default rate!)
✅ loan_intent_EDUCATION (coef=-0.33): Confirms RQ2 finding
   → EDUCATION loans also reduce default risk

LIMITATION:
────────────
This model prioritizes overall accuracy over detecting defaults specifically.
A bank would prefer HIGHER default recall even at the cost of lower accuracy.
Solution: Use class_weight='balanced' parameter in future iterations.
""")

In [ ]:
df.to_csv('fintech_data_final.csv', index=False, encoding='utf-8')